# s3_ihme_gbd_ingest


In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "boto3"], check=True)

import boto3
from pathlib import Path
import os
from pyspark.sql import functions as F

BUCKET = "mortality-analytics-semi2"
REGION = "us-east-1"

def _client():
    access_key = dbutils.secrets.get(scope="semis2_scope", key="AWS_ACCESS_KEY_ID")
    secret_key = dbutils.secrets.get(scope="semis2_scope", key="AWS_SECRET_ACCESS_KEY")

    return boto3.client(
        "s3",
        region_name=REGION,
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key

    )


def upload_file(local_path: str, s3_key: str) -> None:
    _client().upload_file(local_path, BUCKET, s3_key)
    print(f"Uploaded: {local_path} → s3://{BUCKET}/{s3_key}")


def download_file(s3_key: str, destination: str) -> None:
    Path(destination).parent.mkdir(parents=True, exist_ok=True)
    _client().download_file(BUCKET, s3_key, destination)
    print(f"Downloaded: s3://{BUCKET}/{s3_key} → {destination}")


def list_files(prefix: str = "") -> list[dict]:
    paginator = _client().get_paginator("list_objects_v2")
    results = []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            results.append({
                "key": obj["Key"],
                "size": obj["Size"],
                "modified": str(obj["LastModified"]),
            })
    return results


def test_connection() -> None:
    print(f"Bucket: s3://{BUCKET}")
    files = list_files()
    print(f"Objects: {len(files)}")
    for f in files:
        print(f"  {f['key']} ({f['size']} bytes)")


if __name__ == "__main__":
    test_connection()

    spark.sql("CREATE SCHEMA IF NOT EXISTS sandbox")

    import pandas as pd

    workspace_tmp_dir = "/Workspace/Shared/tmp/ihme"
    os.makedirs(workspace_tmp_dir, exist_ok=True)

    final_path = f"{workspace_tmp_dir}/ihme_gbd_2023_central_america_by_cause.csv"

    download_file("raw/ihme/ihme_gbd_2023_central_america_by_cause.csv", final_path)

    # Databricks Connect: Python descarga al FS del cliente, pero spark.read con
    # scheme `file:` busca en el FS del cluster remoto (donde el archivo no está).
    # Se lee con pandas en el cliente y se envía al cluster vía createDataFrame.
    # CSV de IHME GBD, separador ',', todas como string (inferSchema=False).
    pdf = pd.read_csv(final_path, dtype=str, keep_default_na=False, encoding="utf-8")

    df = (
        spark.createDataFrame(pdf)
        .withColumn("_source_file", F.lit("ihme_gbd_2023_central_america_by_cause.csv"))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

    display(df)

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sandbox.raw_ihme")
